# Session 5 — Persistence Baseline & Trainer Skeleton

I'm reloading the model-ready table from Session 4 and confirming the contract
before I build anything: right shape, all feature/meta/target columns present,
and a first look at how imbalanced the AQI categories are — that skew is exactly
why I'll score classification on weighted-F1, not accuracy.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src import config

features = pd.read_parquet(config.FEATURES_PATH)

print("shape:", features.shape)
print("cities:", features[config.CITY_ID_COL].nunique())
print("date range:", features[config.DATE_COL].min(), "->", features[config.DATE_COL].max())

# The contract: every column Sessions 6-7 will reach for must be present.
expected = (
    config.META_COLS
    + config.FEATURE_COLS
    + [config.TARGET_AQI_COL, config.TARGET_BUCKET_COL]
)
missing = [c for c in expected if c not in features.columns]
assert len(missing) == 0, f"missing expected columns: {missing}"
print("contract OK: all", len(expected), "expected columns present")

# Class balance of the classification target — the case for weighted-F1.
print("\ntarget_bucket balance (share of rows):")
print(features[config.TARGET_BUCKET_COL].value_counts(normalize=True).round(3))

shape: (201725, 30)
cities: 141
date range: 2018-01-08 00:00:00 -> 2023-03-30 00:00:00
contract OK: all 30 expected columns present

target_bucket balance (share of rows):
target_bucket
Moderate        0.352
Satisfactory    0.340
Good            0.137
Poor            0.085
Very Poor       0.074
Severe          0.012
Name: proportion, dtype: float64


## Chronological train/test split

I'm splitting on a single global date — train on everything before 2022-04-01,
test on everything from that date on. This mirrors deployment (forecast forward
from a fixed point) and makes leakage structurally impossible: no training row
shares a date with any test row. I hold out ~12 months so my test set spans
every season, not just one.

In [2]:
# Global chronological cutoff. Train = strictly before; test = on/after.
# Inline for now — promoted to config in Bucket 5 (crystallization).
TEST_CUTOFF = pd.Timestamp("2022-04-01")

is_train = features[config.DATE_COL] < TEST_CUTOFF
train = features.loc[is_train].copy()
test = features.loc[~is_train].copy()

print("train rows:", len(train), f"({len(train) / len(features):.1%})")
print("test  rows:", len(test),  f"({len(test) / len(features):.1%})")
print("train dates:", train[config.DATE_COL].min(), "->", train[config.DATE_COL].max())
print("test  dates:", test[config.DATE_COL].min(),  "->", test[config.DATE_COL].max())

# Leakage wall: no train date may reach the cutoff; no test date may precede it.
assert train[config.DATE_COL].max() < TEST_CUTOFF, "train leaks past the cutoff"
assert test[config.DATE_COL].min() >= TEST_CUTOFF, "test starts before the cutoff"
print("\nno temporal overlap: train ends strictly before test begins")

# Sanity: every city we'll be scored on in test was also seen during train.
train_cities = set(train[config.CITY_ID_COL])
test_cities = set(test[config.CITY_ID_COL])
assert test_cities <= train_cities, "a test city never appears in train"
print(f"cities — train: {len(train_cities)}, test: {len(test_cities)} "
      f"(every test city was seen in train)")

train rows: 155131 (76.9%)
test  rows: 46594 (23.1%)
train dates: 2018-01-08 00:00:00 -> 2022-03-31 00:00:00
test  dates: 2022-04-01 00:00:00 -> 2023-03-30 00:00:00

no temporal overlap: train ends strictly before test begins
cities — train: 141, test: 141 (every test city was seen in train)


## Persistence baseline — regression

This is my falsifiable bar. The naive forecast just copies today's AQI as the
prediction for tomorrow, so I score it as: prediction = today's `aqi`, truth =
`target_aqi`. I report MAE / RMSE / R2 on the test set — that's the exact bar my
Session 6 models have to beat. The RMSE-MAE gap is the regime-change tax:
overnight spikes persistence can never see coming.

In [3]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


def persistence_regression_scores(frame: pd.DataFrame) -> dict[str, float]:
    """Score the naive 'tomorrow = today' AQI forecast on a frame.

    Prediction is each row's own AQI (today); truth is target_aqi (tomorrow).
    Returns MAE, RMSE, and R2, all in AQI units (R2 unitless).
    """
    y_true = frame[config.TARGET_AQI_COL]
    y_pred = frame[config.AQI_COL]            # persistence: today's AQI as-is
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": root_mean_squared_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }


# Today's AQI must be present on every surviving row, or the baseline can't run.
# It is guaranteed: aqi_roll_mean_3 (window=3, min_periods=3) includes day T, so
# any row with a missing today-AQI was already dropped as a structural NaN.
for name, frame in (("train", train), ("test", test)):
    assert frame[config.AQI_COL].isna().sum() == 0, f"{name}: today's AQI has NaNs"

train_scores = persistence_regression_scores(train)
test_scores = persistence_regression_scores(test)

print("persistence baseline — regression (today predicts tomorrow)\n")
print(f"{'metric':<6}{'train':>12}{'test':>12}")
for m in ("mae", "rmse", "r2"):
    print(f"{m:<6}{train_scores[m]:>12.3f}{test_scores[m]:>12.3f}")

gap = test_scores["rmse"] - test_scores["mae"]
print(f"\nRMSE - MAE gap (test): {gap:.2f}")
print("  -> the regime-change tax: days where 'today' was a bad guess for")
print("     tomorrow (Diwali spikes, dust storms, frontal clearances).")

persistence baseline — regression (today predicts tomorrow)

metric       train        test
mae         27.180      24.008
rmse        46.364      41.404
r2           0.746       0.745

RMSE - MAE gap (test): 17.40
  -> the regime-change tax: days where 'today' was a bad guess for
     tomorrow (Diwali spikes, dust storms, frontal clearances).


## Persistence baseline — classification

Same falsifiable-bar idea as regression, but for the health category. The naive
forecast copies today's `aqi_bucket` as tomorrow's prediction. I score it with
weighted-F1 (my official metric) plus accuracy and macro-F1 for contrast — the
accuracy-vs-macro gap exposes how badly persistence handles the rare dangerous
days. The confusion matrix shows exactly where it fails.

In [4]:
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)


def persistence_classification_scores(frame: pd.DataFrame) -> dict[str, float]:
    """Score the naive 'tomorrow's category = today's category' forecast.

    Prediction is each row's own aqi_bucket (today); truth is target_bucket
    (tomorrow). Returns accuracy, weighted-F1, and macro-F1.
    """
    y_true = frame[config.TARGET_BUCKET_COL]
    y_pred = frame[config.AQI_BUCKET_COL]           # persistence: today's category
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


# Today's category must exist on every row (same structural guarantee as AQI).
for name, frame in (("train", train), ("test", test)):
    assert frame[config.AQI_BUCKET_COL].isna().sum() == 0, f"{name}: today's bucket has NaNs"

train_cls = persistence_classification_scores(train)
test_cls = persistence_classification_scores(test)

print("persistence baseline — classification (today's category predicts tomorrow's)\n")
print(f"{'metric':<14}{'train':>10}{'test':>10}")
for m in ("accuracy", "weighted_f1", "macro_f1"):
    print(f"{m:<14}{train_cls[m]:>10.3f}{test_cls[m]:>10.3f}")

# Per-class detail + confusion matrix on the test set.
y_true = test[config.TARGET_BUCKET_COL]
y_pred = test[config.AQI_BUCKET_COL]

print("\nper-class detail (test):")
print(classification_report(y_true, y_pred, labels=config.AQI_CATEGORIES, zero_division=0))

# labels=AQI_CATEGORIES forces CPCB severity order so the matrix reads diagonally.
cm = confusion_matrix(y_true, y_pred, labels=config.AQI_CATEGORIES)
cm_df = pd.DataFrame(cm, index=config.AQI_CATEGORIES, columns=config.AQI_CATEGORIES)
cm_df.index.name = "actual"
cm_df.columns.name = "predicted"
print("confusion matrix (test) — rows = actual tomorrow, cols = today's guess:")
print(cm_df)

persistence baseline — classification (today's category predicts tomorrow's)

metric             train      test
accuracy           0.692     0.711
weighted_f1        0.692     0.711
macro_f1           0.625     0.606

per-class detail (test):
              precision    recall  f1-score   support

        Good       0.77      0.76      0.77      7713
Satisfactory       0.75      0.74      0.74     16288
    Moderate       0.74      0.74      0.74     15725
        Poor       0.44      0.44      0.44      3853
   Very Poor       0.60      0.60      0.60      2802
      Severe       0.34      0.34      0.34       213

    accuracy                           0.71     46594
   macro avg       0.61      0.61      0.61     46594
weighted avg       0.71      0.71      0.71     46594

confusion matrix (test) — rows = actual tomorrow, cols = today's guess:
predicted     Good  Satisfactory  Moderate  Poor  Very Poor  Severe
actual                                                             
Good 

## Preprocessing + CV scaffold

I'm building the leakage-safe machinery the bake-offs plug into: a
ColumnTransformer that median-imputes + scales the numerics and one-hot-encodes
season, all inside a Pipeline so it only ever fits on training folds. I sort the
training frame by date FIRST — otherwise TimeSeriesSplit would split by city
instead of time, since the frame is grouped per city. Then I smoke-test the whole
rig with one LinearRegression to confirm it runs before I pour 9 models in.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# --- Feature groups (promoted to config at crystallization, Bucket 5) ---
NUMERIC_FEATURES = config.FEATURE_COLS              # 21 numeric cols
CATEGORICAL_FEATURES = [config.SEASON_COL]          # season -> one-hot
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# --- The leakage-safe preprocessor ---
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # fills the carried pollutant NaNs
    ("scale", StandardScaler()),                    # mean 0, std 1
])
categorical_pipe = Pipeline([
    ("encode", OneHotEncoder(handle_unknown="ignore")),  # ignore unseen seasons in a fold
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])

# --- THE FIX: sort train by date so row order is chronological for CV ---
train_sorted = train.sort_values(config.DATE_COL).reset_index(drop=True)

X_train = train_sorted[MODEL_FEATURES]
y_train_reg = train_sorted[config.TARGET_AQI_COL]
X_test = test[MODEL_FEATURES]
y_test_reg = test[config.TARGET_AQI_COL]

# --- Time-aware CV: each fold trains on the past, validates on the future ---
tscv = TimeSeriesSplit(n_splits=5)

print("TimeSeriesSplit folds (proving each val block is LATER than its train block):")
dates = train_sorted[config.DATE_COL]
for i, (tr_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
    print(f"  fold {i}: train {dates.iloc[tr_idx].min().date()} -> {dates.iloc[tr_idx].max().date()}"
          f"  |  val {dates.iloc[val_idx].min().date()} -> {dates.iloc[val_idx].max().date()}")

# --- Smoke test: one LinearRegression through the full pipe, train -> test ---
smoke_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LinearRegression()),
])
smoke_pipe.fit(X_train, y_train_reg)
smoke_pred = smoke_pipe.predict(X_test)
smoke_mae = mean_absolute_error(y_test_reg, smoke_pred)

n_out = preprocessor.fit_transform(X_train).shape[1]
print(f"\npreprocessor output width: {n_out} cols  (21 numeric + one-hot season)")
print(f"smoke-test LinearRegression  test MAE: {smoke_mae:.3f}")
print(f"persistence baseline         test MAE: {test_scores['mae']:.3f}")
print("  -> if the model dips under the baseline, the pipe works AND modeling adds value.")

TimeSeriesSplit folds (proving each val block is LATER than its train block):
  fold 1: train 2018-01-08 -> 2019-02-25  |  val 2019-02-25 -> 2019-11-27
  fold 2: train 2018-01-08 -> 2019-11-27  |  val 2019-11-27 -> 2020-07-20
  fold 3: train 2018-01-08 -> 2020-07-20  |  val 2020-07-20 -> 2021-02-20
  fold 4: train 2018-01-08 -> 2021-02-20  |  val 2021-02-20 -> 2021-09-13
  fold 5: train 2018-01-08 -> 2021-09-13  |  val 2021-09-13 -> 2022-03-31

preprocessor output width: 25 cols  (21 numeric + one-hot season)
smoke-test LinearRegression  test MAE: 23.701
persistence baseline         test MAE: 24.008
  -> if the model dips under the baseline, the pipe works AND modeling adds value.


In [6]:
from src.models.baseline import PersistenceBaseline
from src.models.trainer import ModelTrainer

pb = PersistenceBaseline()
trainer = ModelTrainer()

features = pd.read_parquet(config.FEATURES_PATH)
tr, te = trainer.split(features)

print("regression   :", {k: round(v, 3) for k, v in pb.regression_scores(te).items()})
print("classification:", {k: round(v, 3) for k, v in pb.classification_scores(te).items()})
print("train/test rows:", len(tr), len(te))

regression   : {'mae': np.float64(24.008), 'rmse': np.float64(41.404), 'r2': 0.745}
classification: {'accuracy': 0.711, 'weighted_f1': np.float64(0.711), 'macro_f1': np.float64(0.606)}
train/test rows: 155131 46594
